# Benchmark: Critique-Revise vs LLM-as-a-Judge

Offline-friendly benchmark notebook that loads a checked-in fixture and summarizes token cost, latency, and quality trade-offs.

- Default notebook mode: `stub`
- Switch `MODE` to `live` only when credentials are configured.

In [ ]:
MODE = "stub"
FIXTURE_NAME = "benchmark_critique_vs_judge_fixture.json"

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import json

fixture_path = ROOT / "examples" / FIXTURE_NAME
fixture = json.loads(fixture_path.read_text(encoding="utf-8"))
rows = []
for item in fixture["tasks"]:
    rows.append({
        "task": item["task"],
        "critique_quality": item["critique_revise"]["quality_score"],
        "critique_tokens": item["critique_revise"]["total_tokens"],
        "critique_latency": item["critique_revise"]["latency_seconds"],
        "judge_quality": item["llm_judge"]["quality_score"],
        "judge_tokens": item["llm_judge"]["total_tokens"],
        "judge_latency": item["llm_judge"]["latency_seconds"],
    })
rows

In [ ]:
def make_markdown_table(rows):
    headers = [
        "Task",
        "Critique Quality",
        "Critique Tokens",
        "Critique Latency",
        "Judge Quality",
        "Judge Tokens",
        "Judge Latency",
    ]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append(
            "| "
            + " | ".join(
                [
                    row["task"],
                    str(row["critique_quality"]),
                    str(row["critique_tokens"]),
                    str(row["critique_latency"]),
                    str(row["judge_quality"]),
                    str(row["judge_tokens"]),
                    str(row["judge_latency"]),
                ]
            )
            + " |"
        )
    return "\n".join(lines)

summary = {
    "avg_critique_quality": round(sum(row["critique_quality"] for row in rows) / len(rows), 3),
    "avg_critique_tokens": round(sum(row["critique_tokens"] for row in rows) / len(rows), 1),
    "avg_critique_latency": round(sum(row["critique_latency"] for row in rows) / len(rows), 3),
    "avg_judge_quality": round(sum(row["judge_quality"] for row in rows) / len(rows), 3),
    "avg_judge_tokens": round(sum(row["judge_tokens"] for row in rows) / len(rows), 1),
    "avg_judge_latency": round(sum(row["judge_latency"] for row in rows) / len(rows), 3),
}

MARKDOWN_TABLE = make_markdown_table(rows)
print(MARKDOWN_TABLE)
summary